In [0]:
%python
SOURCE_CATALOG = '02_silver'
SOURCE_SCHEMA = 'silver_transform'
TARGET_CATALOG = '03_gold'
TARGET_SCHEMA = 'analytical'

In [0]:
CREATE CATALOG IF NOT EXISTS 03_gold

In [0]:
USE CATALOG 03_gold;
CREATE SCHEMA IF NOT EXISTS analytical;

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.dim_customers AS
SELECT 
  customer_id,
  customer_name,
  customer_type,
  country,
  signup_date
FROM 02_silver.silver_transform.customers_clean

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.dim_products AS
SELECT 
  product_id,
  product_name,
  category,
  cost_price
FROM 02_silver.silver_transform.products_clean

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.dim_regions AS
SELECT 
  region_code,
  region_name,
  country
FROM 02_silver.silver_transform.regions_clean

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.dim_dates AS
SELECT DISTINCT
  invoice_date AS date,
  YEAR(invoice_date) AS year,
  MONTH(invoice_date) AS month,
  DAY(invoice_date) AS day,
  QUARTER(invoice_date) AS quarter,
  DAYOFWEEK(invoice_date) AS day_of_week,
  DATE_FORMAT(invoice_date, 'MMMM') AS month_name,
  DATE_FORMAT(invoice_date, 'EEEE') AS day_name
FROM 02_silver.silver_transform.invoices_clean
UNION
SELECT DISTINCT
  payment_date AS date,
  YEAR(payment_date) AS year,
  MONTH(payment_date) AS month,
  DAY(payment_date) AS day,
  QUARTER(payment_date) AS quarter,
  DAYOFWEEK(payment_date) AS day_of_week,
  DATE_FORMAT(payment_date, 'MMMM') AS month_name,
  DATE_FORMAT(payment_date, 'EEEE') AS day_name
FROM 02_silver.silver_transform.payments_clean

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.cube_sales_analytics AS
SELECT 
  ili.invoice_line_id,
  ili.invoice_id,
  ili.product_id,
  ili.quantity,
  ili.unit_price,
  ili.discount,
  ili.line_amount,
  ili.line_amount * COALESCE(er.rate_to_usd, 1) AS line_amount_usd,
  
  i.customer_id,
  i.invoice_date,
  i.invoice_month,
  i.currency,
  i.region,
  i.invoice_status,
  

  c.customer_name,
  c.customer_type,
  c.country AS customer_country,
  c.signup_date,
  
 
  p.product_name,
  p.category AS product_category,
  p.cost_price,
  

  d.year AS invoice_year,
  d.month AS invoice_month_num,
  d.day AS invoice_day,
  d.quarter AS invoice_quarter,
  d.day_of_week AS invoice_day_of_week,
  d.month_name AS invoice_month_name,
  d.day_name AS invoice_day_name,
  
  COALESCE(pay.payment_count, 0) AS payment_count,
  COALESCE(pay.total_payment_amount, 0) AS total_payment_amount,
  COALESCE(pay.total_payment_amount_usd, 0) AS total_payment_amount_usd,
  COALESCE(pay.payment_methods, ARRAY()) AS payment_methods
  
FROM 02_silver.silver_transform.invoice_line_items_clean ili
INNER JOIN 02_silver.silver_transform.invoices_clean i 
  ON ili.invoice_id = i.invoice_id
LEFT JOIN 02_silver.silver_transform.exchange_rates_clean er 
  ON i.invoice_date = er.date AND i.currency = er.currency
INNER JOIN 03_gold.analytical.dim_customers c 
  ON i.customer_id = c.customer_id
INNER JOIN 03_gold.analytical.dim_products p 
  ON ili.product_id = p.product_id
INNER JOIN 03_gold.analytical.dim_dates d 
  ON i.invoice_date = d.date
LEFT JOIN (
  SELECT 
    pay_tbl.invoice_id,
    COUNT(*) AS payment_count,
    SUM(pay_tbl.payment_amount) AS total_payment_amount,
    SUM(pay_tbl.payment_amount * COALESCE(er_sub.rate_to_usd, 1)) AS total_payment_amount_usd,
    COLLECT_SET(pay_tbl.payment_method) AS payment_methods
  FROM 02_silver.silver_transform.payments_clean pay_tbl
  INNER JOIN 02_silver.silver_transform.invoices_clean inv_sub 
    ON pay_tbl.invoice_id = inv_sub.invoice_id
  LEFT JOIN 02_silver.silver_transform.exchange_rates_clean er_sub
    ON pay_tbl.payment_date = er_sub.date AND inv_sub.currency = er_sub.currency
  GROUP BY pay_tbl.invoice_id
) pay ON i.invoice_id = pay.invoice_id

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.kpi_total_revenue AS
SELECT 
  SUM(line_amount_usd) AS total_revenue_usd,
  COUNT(DISTINCT invoice_id) AS total_invoices,
  COUNT(DISTINCT customer_id) AS total_customers
FROM 03_gold.analytical.cube_sales_analytics
WHERE invoice_status = 'Paid'

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.kpi_quantity_sold AS
SELECT 
  SUM(quantity) AS total_quantity_sold,
  COUNT(DISTINCT product_id) AS products_sold
FROM 03_gold.analytical.cube_sales_analytics
WHERE invoice_status = 'Paid'

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.kpi_avg_invoice_value AS
SELECT 
  AVG(invoice_total) AS avg_invoice_value_usd,
  MIN(invoice_total) AS min_invoice_value_usd,
  MAX(invoice_total) AS max_invoice_value_usd
FROM (
  SELECT 
    invoice_id,
    SUM(line_amount_usd) AS invoice_total
  FROM 03_gold.analytical.cube_sales_analytics
  WHERE invoice_status = 'Paid'
  GROUP BY invoice_id
)

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.kpi_top_customers AS
SELECT 
  customer_id,
  customer_name,
  customer_type,
  customer_country,
  SUM(line_amount_usd) AS total_revenue_usd,
  COUNT(DISTINCT invoice_id) AS total_invoices,
  SUM(quantity) AS total_quantity
FROM 03_gold.analytical.cube_sales_analytics
WHERE invoice_status = 'Paid'
GROUP BY customer_id, customer_name, customer_type, customer_country
ORDER BY total_revenue_usd DESC

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.kpi_top_products AS
SELECT 
  product_id,
  product_name,
  product_category,
  SUM(line_amount_usd) AS total_revenue_usd,
  SUM(quantity) AS total_quantity_sold,
  COUNT(DISTINCT invoice_id) AS total_invoices
FROM 03_gold.analytical.cube_sales_analytics
WHERE invoice_status = 'Paid'
GROUP BY product_id, product_name, product_category
ORDER BY total_revenue_usd DESC

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.kpi_cancellation_rate AS
SELECT 
  COUNT(DISTINCT CASE WHEN invoice_status = 'Cancelled' THEN invoice_id END) AS cancelled_invoices,
  COUNT(DISTINCT invoice_id) AS total_invoices,
  ROUND(COUNT(DISTINCT CASE WHEN invoice_status = 'Cancelled' THEN invoice_id END) * 100.0 / COUNT(DISTINCT invoice_id), 2) AS cancellation_rate_pct
FROM 03_gold.analytical.cube_sales_analytics

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.kpi_discount_impact AS
SELECT 
  SUM(quantity * unit_price * discount) AS total_discount_amount_usd,
  AVG(discount) AS avg_discount_rate,
  COUNT(DISTINCT CASE WHEN discount > 0 THEN invoice_id END) AS invoices_with_discount,
  COUNT(DISTINCT invoice_id) AS total_invoices
FROM 03_gold.analytical.cube_sales_analytics
WHERE invoice_status = 'Paid'

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.kpi_revenue_by_region AS
SELECT 
  region,
  SUM(line_amount_usd) AS total_revenue_usd,
  COUNT(DISTINCT invoice_id) AS total_invoices,
  COUNT(DISTINCT customer_id) AS total_customers,
  SUM(quantity) AS total_quantity
FROM 03_gold.analytical.cube_sales_analytics
WHERE invoice_status = 'Paid'
GROUP BY region
ORDER BY total_revenue_usd DESC

In [0]:
CREATE OR REPLACE TABLE 03_gold.analytical.kpi_invoices_per_customer AS
SELECT 
  customer_id,
  customer_name,
  customer_type,
  COUNT(DISTINCT invoice_id) AS total_invoices,
  SUM(line_amount_usd) AS total_revenue_usd,
  AVG(line_amount_usd) AS avg_order_value_usd
FROM 03_gold.analytical.cube_sales_analytics
WHERE invoice_status = 'Paid'
GROUP BY customer_id, customer_name, customer_type
ORDER BY total_invoices DESC

In [0]:
SELECT 
  CONCAT('$', FORMAT_NUMBER(total_revenue_usd, 2)) AS `Total Revenue (USD)`,
  FORMAT_NUMBER(total_invoices, 0) AS `Total Invoices`,
  FORMAT_NUMBER(total_customers, 0) AS `Total Customers`
FROM 03_gold.analytical.kpi_total_revenue

In [0]:
SELECT 
  customer_id AS `Customer ID`,
  customer_name AS `Customer Name`,
  customer_type AS `Type`,
  customer_country AS `Country`,
  CONCAT('$', FORMAT_NUMBER(total_revenue_usd, 2)) AS `Total Revenue (USD)`,
  FORMAT_NUMBER(total_invoices, 0) AS `Total Invoices`,
  FORMAT_NUMBER(total_quantity, 0) AS `Total Quantity`
FROM 03_gold.analytical.kpi_top_customers 
LIMIT 10

In [0]:
SELECT 
  product_id AS `Product ID`,
  product_name AS `Product Name`,
  product_category AS `Category`,
  CONCAT('$', FORMAT_NUMBER(total_revenue_usd, 2)) AS `Total Revenue (USD)`,
  FORMAT_NUMBER(total_quantity_sold, 0) AS `Quantity Sold`,
  FORMAT_NUMBER(total_invoices, 0) AS `Total Invoices`
FROM 03_gold.analytical.kpi_top_products 
LIMIT 10

In [0]:
SELECT 
  region AS `Region`,
  CONCAT('$', FORMAT_NUMBER(total_revenue_usd, 2)) AS `Total Revenue (USD)`,
  FORMAT_NUMBER(total_invoices, 0) AS `Total Invoices`,
  FORMAT_NUMBER(total_customers, 0) AS `Total Customers`,
  FORMAT_NUMBER(total_quantity, 0) AS `Total Quantity`
FROM 03_gold.analytical.kpi_revenue_by_region

In [0]:
SELECT 
  FORMAT_NUMBER(cancelled_invoices, 0) AS `Cancelled Invoices`,
  FORMAT_NUMBER(total_invoices, 0) AS `Total Invoices`,
  CONCAT(cancellation_rate_pct, '%') AS `Cancellation Rate`
FROM 03_gold.analytical.kpi_cancellation_rate

In [0]:
SELECT 
  FORMAT_NUMBER(total_quantity_sold, 0) AS `Total Quantity Sold`,
  FORMAT_NUMBER(products_sold, 0) AS `Products Sold`
FROM 03_gold.analytical.kpi_quantity_sold

In [0]:
SELECT 
  CONCAT('$', FORMAT_NUMBER(avg_invoice_value_usd, 2)) AS `Avg Invoice Value (USD)`,
  CONCAT('$', FORMAT_NUMBER(min_invoice_value_usd, 2)) AS `Min Invoice Value (USD)`,
  CONCAT('$', FORMAT_NUMBER(max_invoice_value_usd, 2)) AS `Max Invoice Value (USD)`
FROM 03_gold.analytical.kpi_avg_invoice_value

In [0]:
SELECT 
  CONCAT('$', FORMAT_NUMBER(total_discount_amount_usd, 2)) AS `Total Discount Amount (USD)`,
  CONCAT(ROUND(avg_discount_rate * 100, 2), '%') AS `Avg Discount Rate`,
  FORMAT_NUMBER(invoices_with_discount, 0) AS `Invoices with Discount`,
  FORMAT_NUMBER(total_invoices, 0) AS `Total Invoices`
FROM 03_gold.analytical.kpi_discount_impact

In [0]:
SELECT 
  customer_id AS `Customer ID`,
  customer_name AS `Customer Name`,
  customer_type AS `Type`,
  FORMAT_NUMBER(total_invoices, 0) AS `Total Invoices`,
  CONCAT('$', FORMAT_NUMBER(total_revenue_usd, 2)) AS `Total Revenue (USD)`,
  CONCAT('$', FORMAT_NUMBER(avg_order_value_usd, 2)) AS `Avg Order Value (USD)`
FROM 03_gold.analytical.kpi_invoices_per_customer
LIMIT 10